In [17]:
# hello_langchain.py
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()  # reads GOOGLE_API_KEY from .env

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

response = llm.invoke("In one sentence, what is Retrieval-Augmented Generation?")
print(response.content)

Retrieval-Augmented Generation (RAG) enhances large language models by first retrieving relevant information from an external knowledge base and then using that data as context to generate more accurate, grounded, and up-to-date responses.


In [18]:
fake_documents = [
    "TaleemXpress is a student transport booking and management platform originally built for GIK Institute.",
    "The revised TaleemXpress stack uses React 18, Vite, and custom JWT auth, replacing the earlier Next.js and Clerk setup.",
    "The backend uses FastAPI or Express, with NeonDB (PostgreSQL) as the database and Drizzle as the ORM.",
    "AutoDeploy is an n8n-orchestrated deployment platform being built during a TMC internship.",
    "Ask TMC is an AI-powered onboarding knowledge portal with HRM integration, also built during the TMC internship.",
]

In [19]:
def fake_retrieve(query, documents, top_k=2):
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k] if score > 0]

query = "What database does TaleemXpress use?"
retrieved = fake_retrieve(query, fake_documents)
retrieved

['TaleemXpress is a student transport booking and management platform originally built for GIK Institute.',
 'The revised TaleemXpress stack uses React 18, Vite, and custom JWT auth, replacing the earlier Next.js and Clerk setup.']

In [20]:
def build_prompt(query, retrieved_docs):
    context = "\n".join(f"- {doc}" for doc in retrieved_docs)
    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""
    return prompt

prompt = build_prompt(query, retrieved)
print(prompt)

Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
- TaleemXpress is a student transport booking and management platform originally built for GIK Institute.
- The revised TaleemXpress stack uses React 18, Vite, and custom JWT auth, replacing the earlier Next.js and Clerk setup.

Question: What database does TaleemXpress use?

Answer:


In [21]:
response = llm.invoke(prompt)
print(response.content)

I don't know.


In [26]:
query2 = "What is abdullah"
retrieved2 = fake_retrieve(query2, fake_documents)
print("Retrieved:", retrieved2)

prompt2 = build_prompt(query2, retrieved2)
response2 = llm.invoke(prompt2)
print(response2.content)

Retrieved: ['TaleemXpress is a student transport booking and management platform originally built for GIK Institute.', 'AutoDeploy is an n8n-orchestrated deployment platform being built during a TMC internship.']
I don't know.
